# Data Cleaning

## Objective

The objective of this notebook is to clean the Olist E-Commerce datasets before merging them for customer segmentation.

The cleaning process includes:

- Identifying and handling missing values
- Removing duplicate records
- Converting columns to appropriate data types
- Filtering irrelevant records
- Removing unnecessary columns
- Validating the cleaned datasets

The cleaned datasets produced in this notebook will be used for exploratory data analysis, feature engineering, and K-Means clustering.

Import Libraries

In [1]:
import pandas as pd
import numpy as np

Load the Datasets

In [2]:
customers = pd.read_csv(r'C:\Users\Elango s\OneDrive\Documents\Customer-Segmentation\datasets\olist_customers_dataset.csv')
orders = pd.read_csv(r'C:\Users\Elango s\OneDrive\Documents\Customer-Segmentation\datasets\olist_orders_dataset.csv')
order_items = pd.read_csv(r'C:\Users\Elango s\OneDrive\Documents\Customer-Segmentation\datasets\olist_order_items_dataset.csv')
payments = pd.read_csv(r'C:\Users\Elango s\OneDrive\Documents\Customer-Segmentation\datasets\olist_order_payments_dataset.csv')

Verify Dataset Shapes

In [3]:
print("Customers :", customers.shape)
print("Orders :", orders.shape)
print("Order Items :", order_items.shape)
print("Payments :", payments.shape)

Customers : (99441, 5)
Orders : (99441, 8)
Order Items : (112650, 7)
Payments : (103886, 5)


Missing Values

In [4]:
datasets = {
    "Customers": customers,
    "Orders": orders,
    "Order Items": order_items,
    "Payments": payments
}

for name, df in datasets.items():
    print(f"\n{name}")
    print(df.isnull().sum())


Customers
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

Orders
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

Order Items
order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64

Payments
order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64


## Missing Value Analysis

The missing value analysis shows that three of the four datasets are complete and contain no missing values.

Only the **Orders** dataset contains missing values in the following columns:

| Column | Missing Values |
|---------|---------------:|
| order_approved_at | 160 |
| order_delivered_carrier_date | 1,783 |
| order_delivered_customer_date | 2,965 |

These missing values are expected because not every order progresses through the complete delivery process. Orders that were canceled or unavailable may not have approval, shipment, or delivery timestamps.

Since the objective of this project is **customer segmentation**, the missing values should be analyzed based on their relationship with the order status before deciding whether to remove or retain them.

Investigate Missing Values

In [5]:
# Orders with missing approval date
orders.loc[
    orders["order_approved_at"].isnull(),
    "order_status"
].value_counts()

order_status
canceled     141
delivered     14
created        5
Name: count, dtype: int64

In [6]:
# Orders with missing carrier date
orders.loc[
    orders["order_delivered_carrier_date"].isnull(),
    "order_status"
].value_counts()

order_status
unavailable    609
canceled       550
invoiced       314
processing     301
created          5
approved         2
delivered        2
Name: count, dtype: int64

In [7]:
# Orders with missing customer delivery date
orders.loc[
    orders["order_delivered_customer_date"].isnull(),
    "order_status"
].value_counts()

order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64

## Missing Value Handling Strategy

The missing values in the **Orders** dataset were investigated by comparing them with the `order_status` column.

### Findings

- Most missing values belong to orders with statuses such as **canceled**, **unavailable**, **created**, **processing**, **approved**, **invoiced**, and **shipped**.
- Only a very small number of **delivered** orders contain missing timestamps, indicating minor data quality issues.

Since the objective of this project is **customer segmentation based on completed purchases**, only successfully delivered orders will be considered for analysis.

Therefore, instead of imputing missing timestamps, the dataset will be filtered to include only delivered orders. This naturally removes most irrelevant missing values while ensuring that the analysis is based on completed customer transactions.

Filter Delivered Orders

In [8]:
orders_clean = orders[orders["order_status"] == "delivered"].copy()

print("Original Orders :", orders.shape)
print("Cleaned Orders  :", orders_clean.shape)

Original Orders : (99441, 8)
Cleaned Orders  : (96478, 8)


Check Remaining Missing Values

In [9]:
orders_clean.isnull().sum()

order_id                          0
customer_id                       0
order_status                      0
order_purchase_timestamp          0
order_approved_at                14
order_delivered_carrier_date      2
order_delivered_customer_date     8
order_estimated_delivery_date     0
dtype: int64

Remove Remaining Missing Records

In [10]:
orders_clean = orders_clean.dropna().reset_index(drop=True)

print(orders_clean.shape)

(96455, 8)


In [11]:
orders_clean.isnull().sum()

order_id                         0
customer_id                      0
order_status                     0
order_purchase_timestamp         0
order_approved_at                0
order_delivered_carrier_date     0
order_delivered_customer_date    0
order_estimated_delivery_date    0
dtype: int64

Check for Duplicates

In [12]:
print("Customers :", customers.duplicated().sum())
print("Orders :", orders_clean.duplicated().sum())
print("Order Items :", order_items.duplicated().sum())
print("Payments :", payments.duplicated().sum())

Customers : 0
Orders : 0
Order Items : 0
Payments : 0


Convert Data Types

In [13]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

orders_clean[date_columns] = orders_clean[date_columns].apply(pd.to_datetime)

In [14]:
orders_clean[date_columns].dtypes

order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object

## Data Type Conversion

The timestamp columns in the Orders dataset were converted from the `object` data type to the `datetime` data type.

This conversion enables efficient date-based operations such as:

- Calculating customer recency
- Measuring delivery duration
- Extracting year, month, and day information
- Performing time-series analysis

Converting these columns ensures that date calculations can be performed accurately during feature engineering.

Remove Unnecessary Columns

In [15]:
# No columns dropped
customers_clean = customers.copy()

In [16]:
orders_clean = orders_clean.drop(columns=[
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date"
])

In [17]:
order_items_clean = order_items.drop(columns=[
    "seller_id",
    "shipping_limit_date"
])

In [18]:
payments_clean = payments.drop(columns=[
    "payment_sequential"
])

In [19]:
print(customers_clean.columns)
print(orders_clean.columns)
print(order_items_clean.columns)
print(payments_clean.columns)

Index(['customer_id', 'customer_unique_id', 'customer_zip_code_prefix',
       'customer_city', 'customer_state'],
      dtype='object')
Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_estimated_delivery_date'],
      dtype='object')
Index(['order_id', 'order_item_id', 'product_id', 'price', 'freight_value'], dtype='object')
Index(['order_id', 'payment_type', 'payment_installments', 'payment_value'], dtype='object')


## Dataset After Cleaning

After removing missing values, duplicate records, converting data types, and dropping unnecessary columns, the datasets were prepared for further analysis.

### Customers Dataset
- customer_id
- customer_unique_id
- customer_zip_code_prefix
- customer_city
- customer_state

### Orders Dataset
- order_id
- customer_id
- order_status
- order_purchase_timestamp
- order_estimated_delivery_date

### Order Items Dataset
- order_id
- order_item_id
- product_id
- price
- freight_value

### Payments Dataset
- order_id
- payment_type
- payment_installments
- payment_value

These cleaned datasets will be used for Exploratory Data Analysis (EDA), dataset merging, feature engineering (RFM), and K-Means clustering.

In [20]:
import os

# Create the cleaned data folder if it doesn't exist
os.makedirs("../data/cleaned", exist_ok=True)

# Save cleaned datasets
customers_clean.to_csv("../datasets/cleaned/customers_clean.csv", index=False)
orders_clean.to_csv("../datasets/cleaned/orders_clean.csv", index=False)
order_items_clean.to_csv("../datasets/cleaned/order_items_clean.csv", index=False)
payments_clean.to_csv("../datasets/cleaned/payments_clean.csv", index=False)

print("All cleaned datasets have been saved successfully!")

All cleaned datasets have been saved successfully!
